# 🎬 Movie Recommendation System
### Content-Based Recommendation using TF-IDF Vectorizer and Cosine Similarity
---
This project builds a complete **Content-Based Movie Recommendation System** based on movie metadata such as `genres`, `keywords`, `tagline`, `cast`, and `director` using the `movies.csv` dataset.

## 1. Mathematical Foundation: Cosine Similarity
**Cosine similarity** is the cosine of the angle between two non-zero vectors in an n-dimensional space.
It is calculated as the dot product of the two vectors divided by the product of their Euclidean lengths (magnitudes):

$$\text{Cosine Similarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|} = \frac{\sum_{i=1}^n u_i v_i}{\sqrt{\sum_{i=1}^n u_i^2} \sqrt{\sum_{i=1}^n v_i^2}}$$

### Numerical Example (from PRJ Document):
- Let $\text{vec}_1 = [1, 1, 0, 1, 1]$
- Let $\text{vec}_2 = [0, 1, 0, 1, 1]$

$$\text{vec}_1 \cdot \text{vec}_2 = 1\times 0 + 1\times 1 + 0\times 0 + 1\times 1 + 1\times 1 = 0 + 1 + 0 + 1 + 1 = 3$$
$$\|\text{vec}_1\| = \sqrt{1^2 + 1^2 + 0^2 + 1^2 + 1^2} = \sqrt{4} = 2$$
$$\|\text{vec}_2\| = \sqrt{0^2 + 1^2 + 0^2 + 1^2 + 1^2} = \sqrt{3} \approx 1.732$$
$$\text{Cosine Similarity} = \frac{3}{2 \times 1.732} = \frac{3}{3.464} \approx 0.8660$$

In [1]:
from sklearn.metrics.pairwise import cosine_similarity

# Verification of Cosine Similarity calculation
vec1 = [1, 1, 0, 1, 1]
vec2 = [0, 1, 0, 1, 1]

sim_score = cosine_similarity([vec1], [vec2])[0][0]
print(f"Cosine Similarity of vec1 and vec2: {sim_score:.4f}")

Cosine Similarity of vec1 and vec2: 0.8660


## 2. Import Libraries

In [2]:
import numpy as np
import pandas as pd
import difflib
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("All libraries imported successfully!")

All libraries imported successfully!


## 3. Check Directory & Load Dataset

In [3]:
# Check current working directory
print("Current Working Directory:", os.getcwd())

# Load dataset
movies_data = pd.read_csv('movies.csv')
display(movies_data.head())

Current Working Directory: C:\Users\MONU SHARMA\OneDrive\Desktop\new project ritesh


,index,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew,director
0,0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Sam Worthington Zoe Saldana Sigourney Weaver S...,"[{'name': 'Stephen E. Rivkin', 'gender': 0, 'd...",James Cameron
1,1,300000000,Adventure Fantasy Action,http://disney.go.com/disneypictures/pirates/,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Johnny Depp Orlando Bloom Keira Knightley Stel...,"[{'name': 'Dariusz Wolski', 'gender': 2, 'depa...",Gore Verbinski
2,2,245000000,Action Adventure Crime,http://www.sonypictures.com/movies/spectre/,206647,spy based on novel secret agent sequel mi6,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,"[{'name': 'Thomas Newman', 'gender': 2, 'depar...",Sam Mendes
3,3,250000000,Action Crime Drama Thriller,http://www.thedarkknightrises.com/,49026,dc comics crime fighter terrorist secret ident...,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,...,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,Christian Bale Michael Caine Gary Oldman Anne ...,"[{'name': 'Hans Zimmer', 'gender': 2, 'departm...",Christopher Nolan
4,4,260000000,Action Adventure Science Fiction,http://movies.disney.com/john-carter,49529,based on novel mars medallion space travel pri...,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,...,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,Taylor Kitsch Lynn Collins Samantha Morton Wil...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton


In [4]:
# Display the shape of the dataset
print("Dataset Shape:", movies_data.shape)
display(movies_data.shape)

Dataset Shape: (4803, 24)


(4803, 24)

In [5]:
# Display dataset information (columns, data types, non-null counts)
movies_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   index                 4803 non-null   int64  
 1   budget                4803 non-null   int64  
 2   genres                4775 non-null   str    
 3   homepage              1712 non-null   str    
 4   id                    4803 non-null   int64  
 5   keywords              4391 non-null   str    
 6   original_language     4803 non-null   str    
 7   original_title        4803 non-null   str    
 8   overview              4800 non-null   str    
 9   popularity            4803 non-null   float64
 10  production_companies  4803 non-null   str    
 11  production_countries  4803 non-null   str    
 12  release_date          4802 non-null   str    
 13  revenue               4803 non-null   int64  
 14  runtime               4801 non-null   float64
 15  spoken_languages      4803 non-n

In [6]:
# Check for Null/Missing Values in all columns
display(movies_data.isna().sum())

index                      0
budget                     0
genres                    28
homepage                3091
id                         0
keywords                 412
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average               0
vote_count                 0
cast                      43
crew                       0
director                  30
dtype: int64

## 4. Selecting Relevant Features for Recommendation
We select the 5 most informative text features describing each movie:
- `genres`
- `keywords`
- `tagline`
- `cast`
- `director`

In [7]:
selected_features = ['genres', 'keywords', 'tagline', 'cast', 'director']
print("Selected Features:", selected_features)
display(movies_data[selected_features].head())

Selected Features: ['genres', 'keywords', 'tagline', 'cast', 'director']


,genres,keywords,tagline,cast,director
0,Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...,Enter the World of Pandora.,Sam Worthington Zoe Saldana Sigourney Weaver S...,James Cameron
1,Adventure Fantasy Action,ocean drug abuse exotic island east india trad...,"At the end of the world, the adventure begins.",Johnny Depp Orlando Bloom Keira Knightley Stel...,Gore Verbinski
2,Action Adventure Crime,spy based on novel secret agent sequel mi6,A Plan No One Escapes,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,Sam Mendes
3,Action Crime Drama Thriller,dc comics crime fighter terrorist secret ident...,The Legend Ends,Christian Bale Michael Caine Gary Oldman Anne ...,Christopher Nolan
4,Action Adventure Science Fiction,based on novel mars medallion space travel pri...,"Lost in our world, found in another.",Taylor Kitsch Lynn Collins Samantha Morton Wil...,Andrew Stanton


In [8]:
# Check Null Values in selected columns before filling
display(movies_data[selected_features].isna().sum())

genres       28
keywords    412
tagline     844
cast         43
director     30
dtype: int64

## 5. Replacing Null Values with Empty String
To avoid `NaN` concatenation issues, we fill null values in selected features with `""`.

In [9]:
for feature in selected_features:
    movies_data[feature] = movies_data[feature].fillna('')

# Confirm null values are resolved
display(movies_data[selected_features].isna().sum())

genres      0
keywords    0
tagline     0
cast        0
director    0
dtype: int64

## 6. Combining the Selected Features
We combine all 5 text features into a single composite string per movie.

In [10]:
combined_features = (
    movies_data['genres'] + ' ' +
    movies_data['keywords'] + ' ' +
    movies_data['tagline'] + ' ' +
    movies_data['cast'] + ' ' +
    movies_data['director']
)

display(combined_features.head())

0    Action Adventure Fantasy Science Fiction cultu...
1    Adventure Fantasy Action ocean drug abuse exot...
2    Action Adventure Crime spy based on novel secr...
3    Action Crime Drama Thriller dc comics crime fi...
4    Action Adventure Science Fiction based on nove...
dtype: str

## 7. Converting Text Data to Feature Vectors (TF-IDF)
**TF-IDF** (Term Frequency - Inverse Document Frequency):
- **Term Frequency:** $tf(t, d) = \frac{\text{count of } t \text{ in } d}{\text{number of words in } d}$
- **Inverse Document Frequency:** $idf(t) = 1 + \ln\left(\frac{n}{df(t)}\right)$
- **TF-IDF:** $\text{TF-IDF}(t, d) = tf(t, d) \times idf(t)$

In [11]:
vectorizer = TfidfVectorizer()
feature_vectors = vectorizer.fit_transform(combined_features)

print("Shape of TF-IDF feature vectors:", feature_vectors.shape)
display(feature_vectors.shape)

Shape of TF-IDF feature vectors: (4803, 17318)


(4803, 17318)

## 8. Compute Cosine Similarity Matrix
Finding the pairwise cosine similarity score between all movies.

In [12]:
similarity = cosine_similarity(feature_vectors)
print("Cosine Similarity Matrix:")
print(similarity)
print("Shape of Similarity Matrix:", similarity.shape)

Cosine Similarity Matrix:
[[1.         0.07219487 0.037733   ... 0.         0.         0.        ]
 [0.07219487 1.         0.03281499 ... 0.03575545 0.         0.        ]
 [0.037733   0.03281499 1.         ... 0.         0.05389661 0.        ]
 ...
 [0.         0.03575545 0.         ... 1.         0.         0.02651502]
 [0.         0.         0.05389661 ... 0.         1.         0.        ]
 [0.         0.         0.         ... 0.02651502 0.         1.        ]]
Shape of Similarity Matrix: (4803, 4803)


## 9. Finding Close Match & Movie Recommendations
Using `difflib.get_close_matches` to locate the closest movie title in case of spelling mistakes or variations.

In [13]:
# Creating a list of all movie titles
list_of_all_titles = movies_data['title'].tolist()
print(f"Total movie titles: {len(list_of_all_titles)}")

Total movie titles: 4803


In [14]:
# Enter movie name
movie_name = 'Iron Man'

# Finding close matches
find_close_match = difflib.get_close_matches(movie_name, list_of_all_titles)
print("All close matches:", find_close_match)

close_match = find_close_match[0]
print("Closest Match:", close_match)

# Finding the index of the movie
index_of_the_movie = movies_data[movies_data.title == close_match]['index'].values[0]
print("Index of Movie:", index_of_the_movie)

# Get similarity row for this movie
similarity_score = list(enumerate(similarity[index_of_the_movie]))

# Sorting the movies based on similarity score in descending order
sorted_similar_movies = sorted(similarity_score, key=lambda x: x[1], reverse=True)

# Print Top 30 recommended movies
print('Movies suggested for you : \n')
i = 1
for movie in sorted_similar_movies:
    index = movie[0]
    title_from_index = movies_data[movies_data.index == index]['title'].values[0]
    if i < 30:
        print(f"{i}. {title_from_index}")
        i += 1

All close matches: ['Iron Man', 'Iron Man 3', 'Iron Man 2']
Closest Match: Iron Man
Index of Movie: 68
Movies suggested for you : 

1. Iron Man
2. Iron Man 2
3. Iron Man 3
4. Avengers: Age of Ultron
5. The Avengers
6. Captain America: Civil War
7. Captain America: The Winter Soldier
8. Ant-Man
9. X-Men
10. Made
11. X-Men: Apocalypse
12. X2
13. The Incredible Hulk
14. The Helix... Loaded
15. X-Men: First Class
16. X-Men: Days of Future Past
17. Captain America: The First Avenger
18. Kick-Ass 2
19. Guardians of the Galaxy
20. Deadpool
21. Thor: The Dark World
22. G-Force
23. X-Men: The Last Stand
24. Duets
25. Mortdecai
26. The Last Airbender
27. Southland Tales
28. Zathura: A Space Adventure
29. Sky Captain and the World of Tomorrow


## 10. Consolidated Function for Movie Recommendation

In [15]:
def get_movie_recommendations(movie_name, top_n=30):
    list_of_all_titles = movies_data['title'].tolist()
    find_close_match = difflib.get_close_matches(movie_name, list_of_all_titles, n=5, cutoff=0.3)
    
    if not find_close_match:
        print(f"No matching movie found for '{movie_name}'. Please try another title.")
        return
        
    close_match = find_close_match[0]
    print(f"Closest match found: '{close_match}'")
    
    index_of_the_movie = movies_data[movies_data.title == close_match]['index'].values[0]
    similarity_score = list(enumerate(similarity[index_of_the_movie]))
    sorted_similar_movies = sorted(similarity_score, key=lambda x: x[1], reverse=True)
    
    print(f"\nTop {top_n} Movies Suggested for you : \n" + "-"*45)
    i = 1
    for movie in sorted_similar_movies:
        index = movie[0]
        title_from_index = movies_data[movies_data.index == index]['title'].values[0]
        score = movie[1]
        if i < top_n:
            print(f"{i:2d}. {title_from_index:<40} (Score: {score:.4f})")
            i += 1

In [16]:
# Example 1: Testing with 'Avatar'
get_movie_recommendations('Avatar', top_n=20)

Closest match found: 'Avatar'

Top 20 Movies Suggested for you : 
---------------------------------------------
 1. Avatar                                   (Score: 1.0000)
 2. Alien                                    (Score: 0.2495)
 3. Aliens                                   (Score: 0.2484)
 4. Guardians of the Galaxy                  (Score: 0.2451)
 5. Star Trek Beyond                         (Score: 0.2038)
 6. Star Trek Into Darkness                  (Score: 0.2012)
 7. Galaxy Quest                             (Score: 0.1970)
 8. Alien³                                   (Score: 0.1802)
 9. Cargo                                    (Score: 0.1765)
10. Trekkies                                 (Score: 0.1745)
11. Gravity                                  (Score: 0.1744)
12. Moonraker                                (Score: 0.1683)
13. Jason X                                  (Score: 0.1650)
14. Pocahontas                               (Score: 0.1606)
15. Space Cowboys                 

In [17]:
# Example 2: Testing with 'The Dark Knight Rises'
get_movie_recommendations('The Dark Knight Rises', top_n=20)

Closest match found: 'The Dark Knight Rises'

Top 20 Movies Suggested for you : 
---------------------------------------------
 1. The Dark Knight Rises                    (Score: 1.0000)
 2. Batman Begins                            (Score: 0.5661)
 3. The Dark Knight                          (Score: 0.4739)
 4. The Prestige                             (Score: 0.2774)
 5. Batman Returns                           (Score: 0.2400)
 6. Interstellar                             (Score: 0.2064)
 7. Batman & Robin                           (Score: 0.2056)
 8. Batman                                   (Score: 0.2044)
 9. Inception                                (Score: 0.1978)
10. Superman                                 (Score: 0.1823)
11. Legend                                   (Score: 0.1655)
12. Kick-Ass 2                               (Score: 0.1653)
13. Kick-Ass                                 (Score: 0.1625)
14. Child 44                                 (Score: 0.1619)
15. Get Smart      